In [84]:
import requests
from datetime import datetime, date

def get_previous_month_09th(year, month):
    if month == 1:
        prev_year = year - 1
        prev_month = 12
    else:
        prev_year = year
        prev_month = month - 1
    return date(prev_year, prev_month, 9).strftime("%Y-%m-%d")

In [88]:
now = datetime.now()
date_now = now.strftime("%Y-%m-%d")
date_cut = get_previous_month_09th(int(now.strftime("%Y")),5)
date_cut

'2025-04-09'

In [89]:
import requests
from datetime import datetime

now = datetime.now()
# get previous month of current month and set day to 09th

date = now.strftime("%Y-%m-") + "09"

url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    # "query.term": "AREA[LastUpdatePostDate]RANGE[2025-04-09,MAX] AND (AREA[Phase]PHASE1 OR AREA[Phase]PHASE2 OR AREA[Phase]PHASE3 OR AREA[Phase]PHASE4 OR AREA[Phase]EARLY_PHASE1)",
    # "query.term": "AREA[StudyType]INTERVENTIONAL",
    # LastUpdateSubmitDate LastUpdatePostDate
    "query.term": f"AREA[LastUpdatePostDate]RANGE[2025-04-09,MAX] AND AREA[StudyType]INTERVENTIONAL AND (AREA[Phase]PHASE1 OR AREA[Phase]PHASE2 OR AREA[Phase]PHASE3 OR AREA[Phase]PHASE4 OR AREA[Phase]EARLY_PHASE1)",
    # "filter.overallStatus": "NOT_YET_RECRUITING|RECRUITING|ACTIVE_NOT_RECRUITING",
    "countTotal": "true",
    "pageSize": "1000"
}
headers = {
    "accept": "application/json"
}

response = requests.get(url, params=params, headers=headers)

if response.status_code == 200:
    data = response.json()
    # print(data)
else:
    print(f"Error: {response.status_code}")

In [90]:
data.keys()
data['totalCount']

4985

In [23]:
data.keys()

dict_keys(['totalCount', 'studies', 'nextPageToken'])

In [91]:
all_updated_studies = [] 
all_updated_studies.extend(data['studies'])

while 'nextPageToken' in data.keys():

    params["pageToken"] = data["nextPageToken"]
    response = requests.get(url, params=params, headers=headers)

    if response.status_code == 200:
        data = response.json()
        all_updated_studies.extend(data['studies'])
    else:
        print(f"Error: {response.status_code}")

In [92]:
len(all_updated_studies)

4985

In [93]:
import boto3
from boto3.dynamodb.conditions import Key
import psycopg2
from psycopg2.extras import RealDictCursor
import json
import logging
from typing import List, Dict
from dotenv import load_dotenv
import os
import pandas as pd
from tqdm import tqdm
from botocore.exceptions import ClientError

load_dotenv()

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# AWS Lambda function details (replace with your own)
LAMBDA_FUNCTION_NAME = 'clinmatch-dev-metamapParser'  # Your Lambda function name
DYNAMODB_TABLE_NAME = 'clinmatch-AACT-metamap'
AWS_REGION = 'ap-east-1'

def query_dynamodb(table_name, nct_id):
    """
    Queries a DynamoDB table for items with a specific partition key (nct_id)
    and a sort key (updated_at) greater than or equal to a specified value.

    :param table_name: The name of the DynamoDB table
    :param nct_id: The value of the partition key to query
    :param updated_at_min: The minimum value for the sort key
    :return: A list of items matching the query
    """
    dynamodb = boto3.Session(aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'), aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')).resource('dynamodb', region_name=AWS_REGION)

    table = dynamodb.Table(table_name)
    
    try:
        # Perform the query using KeyConditionExpression
        response = table.query(
            KeyConditionExpression=Key('nct_id').eq(nct_id)
        )
        
        # Extract the items from the response
        items = response['Items']
        
        return items
    except Exception as e:
        # Print error message if something goes wrong
        print(f"An error occurred: {e}")
        return []
    
def upload_to_s3(s3, bucket_name, file_key, json_data): 

    # Upload the JSON file
    
    # print(f"Uploading JSON file to: {file_key}")
    try:
        s3.put_object(Bucket=bucket_name, Key=file_key, Body=json_data, ContentType='application/json')
        # print(f"Uploaded JSON file to: {file_key}")
    except Exception as e:
        print(f"Error uploading JSON file: {e}")


In [58]:
len(all_updated_studies)

4985

In [34]:
all_updated_studies[0].keys()

dict_keys(['protocolSection', 'derivedSection', 'hasResults'])

In [49]:
all_updated_studies[0]['protocolSection']['identificationModule']

{'nctId': 'NCT05154994',
 'orgStudyIdInfo': {'id': 'HCI143952'},
 'secondaryIdInfos': [{'id': 'NCI-2021-12484',
   'type': 'REGISTRY',
   'domain': 'CTRP (Clinical Trial Reporting Program)'},
  {'id': 'HCI143952',
   'type': 'OTHER',
   'domain': 'Huntsman Cancer Institute/University of Utah'},
  {'id': 'P30CA042014',
   'type': 'NIH',
   'link': 'https://reporter.nih.gov/quickSearch/P30CA042014'}],
 'organization': {'fullName': 'University of Utah', 'class': 'OTHER'},
 'briefTitle': 'Tremelimumab + Durvalumab(MEDI4736)+ Belinostat in Urothelial Carcinoma',
 'officialTitle': 'RESOLVE : A Phase I Trial of Tremelimumab + Durvalumab(MEDI4736)+ Belinostat in Urothelial Carcinoma',
 'acronym': 'RESOLVE'}

In [ ]:
all_updated_studies[0]['protocolSection'].keys()
all_updated_studies[0]['protocolSection']['statusModule']['overallStatus']#['identificationModule']

update_cnt = 0
remove_cnt = 0
add_cnt = 0
do_nothing_cnt = 0

s3 = boto3.Session(aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'), aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')).client('s3', region_name=AWS_REGION)

# Define the bucket name and prefix
bucket_name = 'codex-data-medical'
prefix = 'aact-metamap-monthly-updates/'

# get date.now() in yyyy-mm-dd format
from datetime import datetime
now = datetime.now()
date_folder = now.strftime("%Y-%m-%d")
date_folder_key = f"{prefix}{date_folder}/"

try:
    s3.put_object(Bucket=bucket_name, Key=date_folder_key)
    print(f"Root folder created: {date_folder_key}")
except Exception as e:
    print(f"Error creating folder: {e}")

folders = ['update', 'remove', 'add', 'do_nothing']

for folder in folders:
    # Create the folder object
    folder_key = f"{date_folder_key}{folder}/"
    print(f"Creating folder: {folder_key}")
    try:
        s3.put_object(Bucket=bucket_name, Key=folder_key)
        print(f"Folder created: {folder_key}")
    except Exception as e:
        print(f"Error creating folder: {e}")

for updated_study in tqdm(all_updated_studies):
    nct_id = updated_study['protocolSection']['identificationModule']['nctId']
    status = updated_study['protocolSection']['statusModule']['overallStatus']

    items = query_dynamodb(DYNAMODB_TABLE_NAME, nct_id)
 
    if items and status in ["ACTIVE_NOT_RECRUITING", "RECRUITING", "ACTIVE_NOT_RECRUITING"]:
        # Update DynamoDB item
        update_cnt+=1
        file_key = f"{date_folder_key}update/{nct_id}.json"
        upload_to_s3(s3, bucket_name, file_key, json.dumps(updated_study))

    elif items and status not in ["ACTIVE_NOT_RECRUITING", "RECRUITING", "ACTIVE_NOT_RECRUITING"]:
        # Remove DynamoDB item
        remove_cnt+=1
        file_key = f"{date_folder_key}remove/{nct_id}.json"
        upload_to_s3(s3, bucket_name, file_key, json.dumps(updated_study))
    
    elif len(items) == 0 and status in ["ACTIVE_NOT_RECRUITING", "RECRUITING", "ACTIVE_NOT_RECRUITING"]:
        # Add new DynamoDB item
        add_cnt+=1
        file_key = f"{date_folder_key}add/{nct_id}.json"
        upload_to_s3(s3, bucket_name, file_key, json.dumps(updated_study))

    else: # len(items) == 0 and status not in ["ACTIVE_NOT_RECRUITING", "RECRUITING", "ACTIVE_NOT_RECRUITING"]
        do_nothing_cnt+=1
        file_key = f"{date_folder_key}do_nothing/{nct_id}.json"
        upload_to_s3(s3, bucket_name, file_key, json.dumps(updated_study))

file_key = f"{date_folder_key}metadata.json"
metadata = {
    "update_time": now.strftime("%Y-%m-%d %H:%M:%S"),
    "aact_update_total ": len(all_updated_studies), 
    "update_cnt": update_cnt, 
    "remove_cnt": remove_cnt, 
    "add_cnt": add_cnt, 
    "do_nothing_cnt": do_nothing_cnt
}
upload_to_s3(s3, bucket_name, file_key, json.dumps(metadata))


Root folder created: aact-metamap-monthly-updates/2025-04-28/
Creating folder: aact-metamap-monthly-updates/2025-04-28/update/
Folder created: aact-metamap-monthly-updates/2025-04-28/update/
Creating folder: aact-metamap-monthly-updates/2025-04-28/remove/
Folder created: aact-metamap-monthly-updates/2025-04-28/remove/
Creating folder: aact-metamap-monthly-updates/2025-04-28/add/
Folder created: aact-metamap-monthly-updates/2025-04-28/add/
Creating folder: aact-metamap-monthly-updates/2025-04-28/do_nothing/
Folder created: aact-metamap-monthly-updates/2025-04-28/do_nothing/


100%|██████████| 4985/4985 [14:47<00:00,  5.62it/s]  


In [95]:
s3 = boto3.Session(aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'), aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')).client('s3', region_name=AWS_REGION)

# Define the bucket name and prefix
bucket_name = 'codex-data-medical'
prefix = 'aact-metamap-monthly-updates/'

# get date.now() in yyyy-mm-dd format
from datetime import datetime
now = datetime.now()
date_folder = now.strftime("%Y-%m-%d")
date_folder_key = f"{prefix}{date_folder}/"

try:
    s3.put_object(Bucket=bucket_name, Key=date_folder_key)
    print(f"Root folder created: {date_folder_key}")
except Exception as e:
    print(f"Error creating folder: {e}")

file_key = f"{date_folder_key}all_updated_trials.json"
upload_to_s3(s3, bucket_name, file_key, json.dumps(all_updated_studies))

file_key = f"{date_folder_key}metadata_{now.strftime("%Y-%m-%d")}.json"
metadata = {
    "update_time": now.strftime("%Y-%m-%d %H:%M:%S"),
    "date_cut": "2025-04-09",
    "aact_update_total ": len(all_updated_studies)
}
upload_to_s3(s3, bucket_name, file_key, json.dumps(metadata))


Root folder created: aact-metamap-monthly-updates/2025-04-28/


In [97]:
6000/34073

0.17609250726381592